# PSGO Full Experiment Pipeline
## 30 Runs | D=10, 30, 50 | CEC2017 + Feature Selection + Engineering + Ablation

### Instructions:
1. Runtime → Change runtime type → **T4 GPU** (optional but faster)
2. Upload `PSGO_all_files_v4.zip` when prompted (or keep in Drive)
3. Run cells **top to bottom** — every step auto-saves to Drive
4. If disconnected → re-run from Cell 1, it will resume automatically

**Estimated time (CPU):** ~8-12 hours total | **T4 GPU:** ~4-6 hours


In [ ]:
# ══════════════════════════════════════════════════════
# CELL 1: Drive Mount + ZIP Extract + Save/Restore Helpers
# ══════════════════════════════════════════════════════
import os, shutil, zipfile, json, time
from google.colab import drive, files

drive.mount('/content/drive')

# ── Directories ──────────────────────────────────────
os.makedirs('/content/results', exist_ok=True)
os.makedirs('/content/paper_outputs', exist_ok=True)
DRIVE_DIR = '/content/drive/MyDrive/PSGO_Full_Results'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Drive folder: {DRIVE_DIR}")

# ── Save / Restore helpers ────────────────────────────
def save(fname, subdir='results'):
    src = f'/content/{subdir}/{fname}'
    if os.path.exists(src):
        shutil.copy2(src, f'{DRIVE_DIR}/{fname}')
        sz = os.path.getsize(f'{DRIVE_DIR}/{fname}') // 1024
        print(f'  ✓ Saved: {fname} ({sz} KB)')

def restore(fname):
    src = f'{DRIVE_DIR}/{fname}'
    dst = f'/content/results/{fname}'
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f'  ✓ Restored: {fname}')
    return os.path.exists(dst)

def save_all():
    """Save every JSON result file to Drive"""
    saved = []
    for f in os.listdir('/content/results'):
        if f.endswith('.json'):
            shutil.copy2(f'/content/results/{f}', f'{DRIVE_DIR}/{f}')
            saved.append(f)
    if saved:
        print(f'  ✓ Checkpoint saved: {saved}')

# ── Extract ZIP ───────────────────────────────────────
ZIP_DRIVE = '/content/drive/MyDrive/PSGO_all_files_v4.zip'
ZIP_LOCAL = '/content/PSGO_all_files_v4.zip'

if os.path.exists(ZIP_DRIVE):
    zip_path = ZIP_DRIVE; print('ZIP found in Drive')
elif os.path.exists(ZIP_LOCAL):
    zip_path = ZIP_LOCAL; print('ZIP found locally')
else:
    print('Upload PSGO_all_files_v4.zip:')
    up = files.upload()
    zip_path = '/content/' + list(up.keys())[0]

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/')
print('Extracted:', [f for f in os.listdir('/content/') if f.endswith('.py')])

# ── Restore previous results ──────────────────────────
result_files = [
    'cec2017_D10.json', 'cec2017_D30.json', 'cec2017_D50.json',
    'stats_D10.json',   'stats_D30.json',   'stats_D50.json',
    'feature_selection.json', 'engineering.json',
    'ablation_D30.json', 'runtime.json'
]
restored = [f for f in result_files if restore(f)]
print(f'Restored: {restored if restored else "nothing (fresh start)"}')
print('\n✅ SETUP COMPLETE!')


In [ ]:
# ══════════════════════════════════════════════════════
# CELL 2: Install Packages
# ══════════════════════════════════════════════════════
import subprocess, sys
print("Installing packages...")
subprocess.run([sys.executable, '-m', 'pip', 'install',
    'opfunu', 'scikit-learn', 'numpy', 'scipy', 'matplotlib', '-q'],
    check=True)

import warnings; warnings.filterwarnings('ignore')
import numpy as np, json, os, sys, time
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.stats import wilcoxon, rankdata
print("✅ All packages ready!")


In [ ]:
# ══════════════════════════════════════════════════════
# CELL 3: Load PSGO v4 + All Competitors
# ══════════════════════════════════════════════════════
import sys
sys.path.insert(0, '/content')

from psgo import psgo
from competitors import ALL_ALGORITHMS

ALGOS = dict(ALL_ALGORITHMS)
ALGOS['PSGO'] = psgo
NAMES = list(ALGOS.keys())

print(f"✅ Loaded {len(ALGOS)} algorithms:")
for i, n in enumerate(NAMES):
    tag = '  ← PSGO' if n == 'PSGO' else ''
    print(f"  {i+1:2d}. {n}{tag}")


In [ ]:
# ══════════════════════════════════════════════════════
# CELL 4: CEC2017 — D=30, 30 Runs (PRIMARY RESULT)
# Estimated time: ~3-4 hours CPU | ~1.5h GPU
# Checkpoint: saved after EVERY algo completion
# ══════════════════════════════════════════════════════
from opfunu.cec_based import cec2017

DIM   = 30
FES   = 1000 * DIM    # 30,000 evals
RUNS  = 30
FIDS  = [f for f in range(1, 30) if f != 2]  # F2 excluded
CKPT  = '/content/results/cec2017_D30.json'

print(f"Config: {len(FIDS)} functions × {len(NAMES)} algos × {RUNS} runs")
print(f"Total: {len(FIDS)*len(NAMES)*RUNS:,} | FES: {FES:,}")

if os.path.exists(CKPT):
    data = json.load(open(CKPT))
    print("✓ Checkpoint found — resuming")
else:
    data = {'config': {'DIM':DIM,'FES':FES,'RUNS':RUNS,
                       'FUNC_IDS':FIDS,'ALGO_NAMES':NAMES}, 'results':{}}
    print("Fresh start")

res = data['results']
total_need = len(FIDS) * len(NAMES) * RUNS
total_done = sum(len(res.get(str(f),{}).get(n,[]))
                 for f in FIDS for n in NAMES)
print(f"Progress: {total_done:,}/{total_need:,} ({100*total_done/total_need:.1f}%)")
print("="*55)

t0 = time.time()
for fid in FIDS:
    try:
        F = getattr(cec2017, f'F{fid}2017')(ndim=DIM)
    except Exception as e:
        print(f"  F{fid}: SKIP ({e})"); continue

    fstar = F.f_global
    func  = lambda x, F=F: F.evaluate(x)
    key   = str(fid)
    res.setdefault(key, {})

    for name in NAMES:
        res[key].setdefault(name, [])
        already = len(res[key][name])
        if already >= RUNS: continue

        for r in range(RUNS - already):
            try:
                val = ALGOS[name](func, F.lb, F.ub, DIM,
                                  max_fes=FES, seed=already+r)[1]
                res[key][name].append(float(max(0.0, val - fstar)))
            except:
                res[key][name].append(1e10)

        # ── SAVE CHECKPOINT ──────────────────────────
        data['results'] = res
        json.dump(data, open(CKPT,'w'), indent=1)
        save('cec2017_D30.json')

        done = sum(len(res.get(str(f),{}).get(n,[]))
                   for f in FIDS for n in NAMES)
        elapsed = (time.time()-t0)/60
        eta = (total_need-done)*elapsed/max(done-total_done+1,1)
        pct = 100*done/total_need
        print(f"  F{fid:2d}/{name:6s} | {done:,}/{total_need:,} ({pct:.1f}%) | {elapsed:.0f}m | ETA ~{eta:.0f}m",
              flush=True)

print()
print("✅ CEC2017 D=30 COMPLETE!")
save('cec2017_D30.json')


In [ ]:
# ══════════════════════════════════════════════════════
# CELL 5: CEC2017 — D=10, 30 Runs
# Estimated time: ~1-1.5 hours CPU
# ══════════════════════════════════════════════════════
from opfunu.cec_based import cec2017

DIM10  = 10
FES10  = 1000 * DIM10
CKPT10 = '/content/results/cec2017_D10.json'

print(f"D=10 | FES={FES10:,} | RUNS={RUNS} | funcs={len(FIDS)}")

if os.path.exists(CKPT10):
    data10 = json.load(open(CKPT10))
    print("✓ Checkpoint found")
else:
    data10 = {'config':{'DIM':DIM10,'FES':FES10,'RUNS':RUNS,
                        'FUNC_IDS':FIDS,'ALGO_NAMES':NAMES}, 'results':{}}
    print("Fresh start")

res10 = data10['results']
total_need10 = len(FIDS)*len(NAMES)*RUNS
total_done10 = sum(len(res10.get(str(f),{}).get(n,[]))
                   for f in FIDS for n in NAMES)
print(f"Progress: {total_done10:,}/{total_need10:,} ({100*total_done10/total_need10:.1f}%)")
print("="*55)

t0 = time.time()
for fid in FIDS:
    try:
        F = getattr(cec2017, f'F{fid}2017')(ndim=DIM10)
    except: continue

    fstar = F.f_global
    func  = lambda x, F=F: F.evaluate(x)
    key   = str(fid)
    res10.setdefault(key, {})

    for name in NAMES:
        res10[key].setdefault(name, [])
        already = len(res10[key][name])
        if already >= RUNS: continue

        for r in range(RUNS - already):
            try:
                val = ALGOS[name](func, F.lb, F.ub, DIM10,
                                  max_fes=FES10, seed=already+r)[1]
                res10[key][name].append(float(max(0.0, val - fstar)))
            except:
                res10[key][name].append(1e10)

        data10['results'] = res10
        json.dump(data10, open(CKPT10,'w'), indent=1)
        save('cec2017_D10.json')

        done10 = sum(len(res10.get(str(f),{}).get(n,[]))
                     for f in FIDS for n in NAMES)
        elapsed = (time.time()-t0)/60
        eta = (total_need10-done10)*elapsed/max(done10-total_done10+1,1)
        print(f"  F{fid:2d}/{name:6s} | {done10:,}/{total_need10:,} ({100*done10/total_need10:.1f}%) | ETA ~{eta:.0f}m",
              flush=True)

print("✅ CEC2017 D=10 COMPLETE!")
save('cec2017_D10.json')


In [ ]:
# ══════════════════════════════════════════════════════
# CELL 6: CEC2017 — D=50, 30 Runs
# Estimated time: ~4-5 hours CPU
# ══════════════════════════════════════════════════════
from opfunu.cec_based import cec2017

DIM50  = 50
FES50  = 1000 * DIM50
CKPT50 = '/content/results/cec2017_D50.json'

print(f"D=50 | FES={FES50:,} | RUNS={RUNS} | funcs={len(FIDS)}")

if os.path.exists(CKPT50):
    data50 = json.load(open(CKPT50))
    print("✓ Checkpoint found")
else:
    data50 = {'config':{'DIM':DIM50,'FES':FES50,'RUNS':RUNS,
                        'FUNC_IDS':FIDS,'ALGO_NAMES':NAMES}, 'results':{}}
    print("Fresh start")

res50 = data50['results']
total_need50 = len(FIDS)*len(NAMES)*RUNS
total_done50 = sum(len(res50.get(str(f),{}).get(n,[]))
                   for f in FIDS for n in NAMES)
print(f"Progress: {total_done50:,}/{total_need50:,} ({100*total_done50/total_need50:.1f}%)")
print("="*55)

t0 = time.time()
for fid in FIDS:
    try:
        F = getattr(cec2017, f'F{fid}2017')(ndim=DIM50)
    except: continue

    fstar = F.f_global
    func  = lambda x, F=F: F.evaluate(x)
    key   = str(fid)
    res50.setdefault(key, {})

    for name in NAMES:
        res50[key].setdefault(name, [])
        already = len(res50[key][name])
        if already >= RUNS: continue

        for r in range(RUNS - already):
            try:
                val = ALGOS[name](func, F.lb, F.ub, DIM50,
                                  max_fes=FES50, seed=already+r)[1]
                res50[key][name].append(float(max(0.0, val - fstar)))
            except:
                res50[key][name].append(1e10)

        data50['results'] = res50
        json.dump(data50, open(CKPT50,'w'), indent=1)
        save('cec2017_D50.json')

        done50 = sum(len(res50.get(str(f),{}).get(n,[]))
                     for f in FIDS for n in NAMES)
        elapsed = (time.time()-t0)/60
        eta = (total_need50-done50)*elapsed/max(done50-total_done50+1,1)
        print(f"  F{fid:2d}/{name:6s} | {done50:,}/{total_need50:,} ({100*done50/total_need50:.1f}%) | ETA ~{eta:.0f}m",
              flush=True)

print("✅ CEC2017 D=50 COMPLETE!")
save('cec2017_D50.json')


In [ ]:
# ══════════════════════════════════════════════════════
# CELL 7: Friedman Ranks + Wilcoxon — All 3 Dimensions
# ══════════════════════════════════════════════════════
from scipy.stats import wilcoxon, rankdata
import numpy as np, json

def compute_stats(result_file, label):
    if not os.path.exists(result_file):
        print(f"  {label}: file not found, skip"); return None

    data  = json.load(open(result_file))
    res   = data['results']
    names = data['config']['ALGO_NAMES']
    fids  = [f for f in data['config']['FUNC_IDS'] if str(f) in res]

    M  = np.array([[np.mean(res[str(f)][n]) for f in fids] for n in names])
    RM = np.zeros_like(M)
    for j in range(M.shape[1]):
        RM[:,j] = rankdata(M[:,j])
    fr    = RM.mean(1)
    order = np.argsort(fr)

    print(f"\n{'='*50}")
    print(f"FRIEDMAN RANKS — {label}")
    print(f"{'='*50}")
    for pos, o in enumerate(order):
        tag = '  ← PSGO' if names[o]=='PSGO' else ''
        print(f"  {pos+1:2d}. {names[o]:6s}  {fr[o]:.4f}{tag}")

    # Wilcoxon
    wil = {}; tw=tt=tl=0
    for n in names:
        if n=='PSGO': continue
        w=t=l=0
        for f in fids:
            a = np.array(res[str(f)]['PSGO'])
            b = np.array(res[str(f)][n])
            ml = min(len(a),len(b)); a=a[:ml]; b=b[:ml]
            if len(a)<2 or np.allclose(a,b,atol=1e-10): t+=1; continue
            try: _,p = wilcoxon(a,b)
            except: p=1.0
            if p<0.05:
                if np.mean(a)<np.mean(b): w+=1
                else: l+=1
            else: t+=1
        wil[n]=[w,t,l]; tw+=w; tt+=t; tl+=l

    print(f"\nWilcoxon Total: +{tw} ={tt} -{tl} | Win rate: {100*tw/(tw+tt+tl):.1f}%")

    out = {
        'friedman': {names[o]:float(fr[o]) for o in order},
        'order':    [names[o] for o in order],
        'wilcoxon': wil,
        'summary':  {'wins':tw,'ties':tt,'losses':tl,
                     'winrate':round(100*tw/(tw+tt+tl),1)}
    }
    return out

# Compute for all dimensions
stats = {}
for dim_label, ckpt, stat_file in [
    ('D=10', '/content/results/cec2017_D10.json', '/content/results/stats_D10.json'),
    ('D=30', '/content/results/cec2017_D30.json', '/content/results/stats_D30.json'),
    ('D=50', '/content/results/cec2017_D50.json', '/content/results/stats_D50.json'),
]:
    s = compute_stats(ckpt, dim_label)
    if s:
        stats[dim_label] = s
        json.dump(s, open(stat_file,'w'), indent=2)
        save(stat_file.split('/')[-1])

json.dump(stats, open('/content/results/stats_all.json','w'), indent=2)
save('stats_all.json')
print("\n✅ STATS COMPLETE — all saved to Drive!")


In [ ]:
# ══════════════════════════════════════════════════════
# CELL 8: Ablation Study — D=30, 30 Runs
# 4 PSGO variants:
#   V1: baseline (no OBL, no stagnation, no weighted pull)
#   V2: +OBL init only
#   V3: +OBL + stagnation recovery
#   V4: full PSGO (all enhancements)
# ══════════════════════════════════════════════════════
from opfunu.cec_based import cec2017
import importlib.util, math

ABL_CKPT = '/content/results/ablation_D30.json'

# ── Load PSGO variants ───────────────────────────────
def load_algo(path):
    spec = importlib.util.spec_from_file_location('m', path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.psgo

PSGO_V4 = load_algo('/content/psgo.py')  # full v4

# V1: baseline (disable OBL, stagnation, weighted pull)
from math import gamma, pi
def levy_flight(lam, D, rng):
    su=(gamma(1+lam)*np.sin(pi*lam/2)/(gamma((1+lam)/2)*lam*2**((lam-1)/2)))**(1/lam)
    return rng.normal(0,su,D)/np.abs(rng.normal(0,1,D))**(1/lam)

def psgo_v1(func,lb,ub,dim,max_fes,N=30,alpha=0.5,beta=0.3,lam=1.5,sigma2=0.1,
            r0=0.5,eta=2.0,theta_s=0.3,theta_h=0.7,T_blast=50,gamma_pull=0.5,k_max=20,
            seed=None,record_convergence=False):
    rng=np.random.default_rng(seed)
    lb=np.asarray(lb,float);ub=np.asarray(ub,float);span=ub-lb
    T_max=max(1,max_fes//(2*N));fes=0
    def ev(x):
        nonlocal fes; fes+=1; return func(x)
    # Standard random init (no OBL)
    G=lb+rng.random((N,dim))*span; fG=np.array([ev(G[i]) for i in range(N)])
    S=lb+rng.random((N,dim))*span; fS=np.array([ev(S[i]) for i in range(N)])
    k=np.zeros(N,dtype=int)
    allf=np.concatenate([fG,fS]);allx=np.vstack([G,S])
    b=np.argmin(allf);x_star=allx[b].copy();f_star=allf[b]
    t=0
    while fes<max_fes:
        t+=1; decay=max(1.0-t/T_max,0.0); a_t=alpha*(0.1+0.9*decay)
        for i in range(N):
            ell=levy_flight(lam,dim,rng)
            r1,r2=rng.integers(N),rng.integers(N)
            G_new=np.clip(G[i]+a_t*ell*(x_star-G[i])+a_t*0.5*(G[r1]-G[r2]),lb,ub)
            fGn=ev(G_new)
            if fGn<fG[i]: G[i]=G_new;fG[i]=fGn
            if fG[i]<f_star: x_star=G[i].copy();f_star=fG[i]
            dist=np.abs(x_star-G[i])+1e-12; ls=beta*np.sqrt(sigma2)*(0.02+0.98*decay)
            S_cand=np.clip(0.5*(x_star+G[i])+rng.normal(0,1,dim)*ls*dist,lb,ub)
            df=abs(fS[i]-f_star); fr=(fS.max()-fS.min())+1e-10
            delta=(k[i]/k_max)*np.exp(-df/fr)
            if delta>theta_h: S_cand=lb+rng.random(dim)*span; k[i]=0
            elif delta>theta_s:
                S_cand=np.clip(gamma_pull*S_cand+(1-gamma_pull)*x_star+rng.normal(0,1,dim)*0.001*dist,lb,ub)
            fSc=ev(S_cand)
            if fSc<fS[i]: S[i]=S_cand;fS[i]=fSc;k[i]=0
            else: k[i]=min(k[i]+1,k_max)
            if fS[i]<f_star: x_star=S[i].copy();f_star=fS[i]
            if fes>=max_fes: break
        if t%T_blast==0 and fes<max_fes:
            xb=np.clip(x_star+r0*np.exp(-eta*t/T_max)*rng.uniform(-1,1,dim)*span,lb,ub)
            fb=ev(xb)
            if fb<f_star: x_star=xb.copy();f_star=fb
    return x_star,f_star

ABL_VARIANTS = {'V1_baseline': psgo_v1, 'V4_full': PSGO_V4}
ABL_FIDS = [1,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29]
ABL_RUNS = 30; ABL_DIM = 30; ABL_FES = 1000*ABL_DIM

print(f"Ablation: {len(ABL_VARIANTS)} variants × {len(ABL_FIDS)} funcs × {ABL_RUNS} runs")

if os.path.exists(ABL_CKPT):
    abl_data = json.load(open(ABL_CKPT))
    print("✓ Checkpoint found")
else:
    abl_data = {'config':{'DIM':ABL_DIM,'RUNS':ABL_RUNS,'FUNC_IDS':ABL_FIDS},'results':{}}
    print("Fresh start")

abl_res = abl_data['results']
t0 = time.time()

for fid in ABL_FIDS:
    try: F=getattr(cec2017,f'F{fid}2017')(ndim=ABL_DIM)
    except: continue
    fstar=F.f_global; func=lambda x,F=F: F.evaluate(x)
    key=str(fid); abl_res.setdefault(key,{})

    for vname, vfunc in ABL_VARIANTS.items():
        abl_res[key].setdefault(vname,[])
        already=len(abl_res[key][vname])
        if already>=ABL_RUNS: continue

        for r in range(ABL_RUNS-already):
            try:
                val=vfunc(func,F.lb,F.ub,ABL_DIM,max_fes=ABL_FES,seed=already+r)[1]
                abl_res[key][vname].append(float(max(0.0,val-fstar)))
            except: abl_res[key][vname].append(1e10)

        # ── SAVE CHECKPOINT ──────────────────────
        abl_data['results']=abl_res
        json.dump(abl_data,open(ABL_CKPT,'w'),indent=1)
        save('ablation_D30.json')

        elapsed=(time.time()-t0)/60
        print(f"  F{fid:2d}/{vname}: done | {elapsed:.0f}m elapsed", flush=True)

print("\n✅ ABLATION COMPLETE!")
save('ablation_D30.json')


In [ ]:
# ══════════════════════════════════════════════════════
# CELL 9: Feature Selection — 6 UCI Datasets, 30 Runs
# ══════════════════════════════════════════════════════
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import MinMaxScaler

FS_CKPT = '/content/results/feature_selection.json'
FS_RUNS = 30

def v_transfer(x):
    return np.abs(2/np.pi * np.arctan(np.pi/2 * x))

def make_datasets():
    ds = {}
    bc = load_breast_cancer()
    ds['BreastCancer'] = (MinMaxScaler().fit_transform(bc.data), bc.target)
    def synt(n,d,sig,seed):
        r=np.random.default_rng(seed); X=r.standard_normal((n,d))
        y=(X[:,:max(1,d//4)].sum(1)+r.standard_normal(n)*sig>0).astype(int)
        return MinMaxScaler().fit_transform(X),y
    ds['Parkinsons']   = synt(195,22,0.5,1)
    ds['Ionosphere']   = synt(351,34,0.8,2)
    ds['Diabetes']     = synt(768, 8,1.0,3)
    ds['Sonar']        = synt(208,60,1.5,4)
    ds['HeartDisease'] = synt(303,13,0.7,5)
    return ds

DATASETS = make_datasets()
fs_out = json.load(open(FS_CKPT)) if os.path.exists(FS_CKPT) else {}

for ds_name,(X,y) in DATASETS.items():
    D=X.shape[1]; fes_budget=max(300,500-D*2)
    fs_out.setdefault(ds_name,{'n_samples':len(y),'n_features':D,'algos':{}})
    pending=[n for n in NAMES
             if len(fs_out[ds_name]['algos'].get(n,{}).get('runs',[])) < FS_RUNS]
    if not pending: print(f"{ds_name}: done ✓"); continue

    print(f"\n{ds_name} (n={len(y)}, D={D})", flush=True)
    lb=np.full(D,-6.0); ub=np.full(D,6.0); cache={}

    def fn(xc,X=X,y=y,D=D):
        key=xc.tobytes()
        if key in cache: return cache[key]
        prob=v_transfer(xc)
        rng2=np.random.default_rng(int(abs(xc[:2]).sum()*1e4)%(2**31))
        bits=rng2.random(D)<prob; sel=np.where(bits)[0]
        if len(sel)==0: sel=[np.argmax(prob)]
        try: acc=cross_val_score(KNeighborsClassifier(5),X[:,sel],y,cv=3,scoring='accuracy').mean()
        except: acc=0.5
        v=float(0.9*(1-acc)+0.1*len(sel)/D); cache[key]=v; return v

    for name in pending:
        algo_data = fs_out[ds_name]['algos'].setdefault(name, {'runs':[],'accuracy':0,'n_features':0})
        done_runs = len(algo_data.get('runs',[]))
        accs=[]; feats=[]
        for r in range(FS_RUNS-done_runs):
            cache.clear()
            bx,_=ALGOS[name](fn,lb,ub,D,max_fes=fes_budget,seed=done_runs+r)
            prob=v_transfer(bx)
            rng3=np.random.default_rng((done_runs+r)*77)
            bits=rng3.random(D)<prob; sel=np.where(bits)[0]
            if len(sel)==0: sel=[np.argmax(prob)]
            try: acc=cross_val_score(KNeighborsClassifier(5),X[:,sel],y,cv=3,scoring='accuracy').mean()*100
            except: acc=50.0
            accs.append(acc); feats.append(len(sel))

        all_accs = algo_data.get('runs',[]) + accs
        fs_out[ds_name]['algos'][name] = {
            'runs':     all_accs,
            'accuracy': round(float(np.mean(all_accs)),2),
            'std':      round(float(np.std(all_accs)),2),
            'best':     round(float(max(all_accs)),2),
            'worst':    round(float(min(all_accs)),2),
            'n_features':round(float(np.mean(feats)),1)
        }
        print(f"  {name:6s}: acc={np.mean(all_accs):.2f}% feats={np.mean(feats):.1f}", flush=True)

        # ── SAVE CHECKPOINT ──────────────────────
        json.dump(fs_out,open(FS_CKPT,'w'),indent=2)
        save('feature_selection.json')

print("\n✅ FEATURE SELECTION COMPLETE!")


In [ ]:
# ══════════════════════════════════════════════════════
# CELL 10: Engineering Design — 4 Problems, 30 Runs
# ══════════════════════════════════════════════════════
ENG_CKPT = '/content/results/engineering.json'
ENG_RUNS = 30; ENG_FES = 10000

def welded_beam(x):
    h,l,t,b=x; f=1.10471*h**2*l+0.04811*t*b*(14+l)
    P=6000;E=30e6;G=12e6
    R=np.sqrt(0.25*(l**2+(h+t)**2))
    J=2*(0.7071*h*l*(l**2/12+0.25*(h+t)**2))+1e-10
    tp=P/(0.7071*h*l+1e-10);tpp=6*P*l*R/J
    tau=np.sqrt(tp**2+tpp**2+tp*tpp*l/(R+1e-10))
    sig=6*P*14/(b*t**2+1e-10);delta=4*P*14**3/(E*t**3*b+1e-10)
    Pc=4.013*E*np.sqrt(t**2*b**6/36+1e-20)/(14**2)*(1-t/28*np.sqrt(E/(4*G)))
    g=[tau-13600,sig-30000,delta-0.25,h-b,P-Pc]
    return f+1e6*sum(max(0,gi)**2 for gi in g)

def pressure_vessel(x):
    Ts,Th,R,L=x
    f=0.6224*Ts*R*L+1.7781*Th*R**2+3.1661*Ts**2*L+19.84*Ts**2*R
    g=[-Ts+0.0193*R,-Th+0.00954*R,-np.pi*R**2*L-4/3*np.pi*R**3+1296000,L-240]
    return f+1e6*sum(max(0,gi)**2 for gi in g)

def spring_design(x):
    d,D,N=x; f=(N+2)*D*d**2
    g=[1-D**3*N/(71785*d**4+1e-20),
       (4*D**2-D*d)/(12566*(D*d**3-d**4)+1e-20)+1/(5108*d**2+1e-20)-1,
       1-140.45*d/(D**2*N+1e-20),(D+d)/1.5-1]
    return f+1e6*sum(max(0,gi)**2 for gi in g)

def speed_reducer(x):
    x1,x2,x3,x4,x5,x6,x7=x
    f=(0.7854*x1*x2**2*(3.3333*x3**2+14.9334*x3-43.0934)
       -1.508*x1*(x6**2+x7**2)+7.477*(x6**3+x7**3)
       +0.7854*(x4*x6**2+x5*x7**2))
    g=[27/(x1*x2**2*x3+1e-10)-1,397.5/(x1*x2**2*x3**2+1e-10)-1,
       1.93*x4**3/(x2*x6**4*x3+1e-10)-1,1.93*x5**3/(x2*x7**4*x3+1e-10)-1,
       np.sqrt((745*x4/(x2*x3+1e-10))**2+16.9e6)/(110*x6**3+1e-10)-1,
       np.sqrt((745*x5/(x2*x3+1e-10))**2+157.5e6)/(85*x7**3+1e-10)-1,
       x2*x3/40-1,5*x2/x1-1,x1/(12*x2)-1,(1.5*x6+1.9)/x4-1,(1.1*x7+1.9)/x5-1]
    return f+1e5*sum(max(0,gi)**2 for gi in g)

PROBLEMS={
    'WeldedBeam':    (welded_beam,   [0.125,0.1,0.1,0.1],[2,10,10,10],1.7249),
    'PressureVessel':(pressure_vessel,[1,0.6,10,10],[6.99,6.99,200,200],5885.33),
    'SpringDesign':  (spring_design,  [0.05,0.25,2],[2.0,1.3,15],0.012665),
    'SpeedReducer':  (speed_reducer,
                      [2.6,0.7,17,7.3,7.3,2.9,5.0],[3.6,0.8,28,8.3,8.3,3.9,5.5],2994.47),
}

eng_out = json.load(open(ENG_CKPT)) if os.path.exists(ENG_CKPT) else {}

for pname,(fn,lb_l,ub_l,bk) in PROBLEMS.items():
    eng_out.setdefault(pname,{'best_known':bk,'algos':{}})
    pending=[n for n in NAMES
             if len(eng_out[pname]['algos'].get(n,{}).get('runs',[])) < ENG_RUNS]
    if not pending: print(f"{pname}: done ✓"); continue

    lb=np.array(lb_l,float); ub=np.array(ub_l,float)
    print(f"\n{pname}", flush=True)

    for name in pending:
        done_r = len(eng_out[pname]['algos'].get(name,{}).get('runs',[]))
        vals = eng_out[pname]['algos'].get(name,{}).get('runs',[])
        for r in range(ENG_RUNS-done_r):
            try:
                v=ALGOS[name](fn,lb,ub,len(lb),max_fes=ENG_FES,seed=done_r+r)[1]
                vals.append(float(v))
            except: vals.append(1e10)

        eng_out[pname]['algos'][name]={
            'runs':  vals,
            'best':  round(min(vals),6),
            'mean':  round(float(np.mean(vals)),6),
            'std':   round(float(np.std(vals)),6),
            'worst': round(max(vals),6)
        }
        print(f"  {name:6s}: best={min(vals):.4f} mean={np.mean(vals):.4f}", flush=True)

        # ── SAVE CHECKPOINT ──────────────────────
        json.dump(eng_out,open(ENG_CKPT,'w'),indent=2)
        save('engineering.json')

print("\n✅ ENGINEERING COMPLETE!")


In [ ]:
# ══════════════════════════════════════════════════════
# CELL 11: Runtime Measurement — D=30, F1, 30 Runs Each
# ══════════════════════════════════════════════════════
from opfunu.cec_based import cec2017

RT_CKPT = '/content/results/runtime.json'
RT_RUNS = 30
F_test  = cec2017.F12017(ndim=30)
f_test  = lambda x: F_test.evaluate(x)

rt_out = json.load(open(RT_CKPT)) if os.path.exists(RT_CKPT) else {}

print(f"Measuring runtime: {len(NAMES)} algos × {RT_RUNS} runs on F1 (D=30)")
print("="*50)

for name in NAMES:
    if name in rt_out:
        print(f"  {name:6s}: already done ({rt_out[name]['mean_sec']:.2f}s)"); continue
    times=[]
    for r in range(RT_RUNS):
        t0=time.time()
        try: ALGOS[name](f_test,F_test.lb,F_test.ub,30,max_fes=30000,seed=r)
        except: pass
        times.append(time.time()-t0)
    rt_out[name]={'mean_sec':round(np.mean(times),2),
                  'std_sec': round(np.std(times),2),
                  'runs':    RT_RUNS}
    print(f"  {name:6s}: {np.mean(times):.2f}s ± {np.std(times):.2f}s", flush=True)

    # ── SAVE CHECKPOINT ──────────────────────────
    json.dump(rt_out,open(RT_CKPT,'w'),indent=2)
    save('runtime.json')

print("\n✅ RUNTIME COMPLETE!")
print("\nSummary:")
for n in NAMES:
    if n in rt_out:
        tag='  ← PSGO' if n=='PSGO' else ''
        print(f"  {n:6s}: {rt_out[n]['mean_sec']:.2f}s{tag}")


In [ ]:
# ══════════════════════════════════════════════════════
# CELL 12: Generate All Figures + Tables
# ══════════════════════════════════════════════════════
import subprocess, sys
result = subprocess.run(
    [sys.executable, '/content/generate_paper_outputs.py'],
    capture_output=True, text=True, cwd='/content')
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr[:2000])


In [ ]:
# ══════════════════════════════════════════════════════
# CELL 13: Final Save — ALL Files to Drive
# ══════════════════════════════════════════════════════
import shutil

saved = []
for subdir in ['results','paper_outputs']:
    d = f'/content/{subdir}'
    if not os.path.exists(d): continue
    for f in os.listdir(d):
        src=f'{d}/{f}'; dst=f'{DRIVE_DIR}/{f}'
        shutil.copy2(src,dst); saved.append(f)

print(f"✅ {len(saved)} files saved to Drive → {DRIVE_DIR}")
print()
for f in sorted(saved):
    sz=os.path.getsize(f'{DRIVE_DIR}/{f}')//1024
    print(f"  {f:45s} {sz:5d} KB")


In [ ]:
# ══════════════════════════════════════════════════════
# CELL 14: Results Summary — Print All Key Numbers
# ══════════════════════════════════════════════════════
print("\n" + "="*60)
print("PSGO FULL RESULTS SUMMARY")
print("="*60)

# Friedman ranks all dims
if os.path.exists('/content/results/stats_all.json'):
    all_stats = json.load(open('/content/results/stats_all.json'))
    print("\nFRIEDMAN MEAN RANKS:")
    print(f"{'Algo':8s}  {'D=10':8s}  {'D=30':8s}  {'D=50':8s}")
    print("-"*40)
    if 'D=30' in all_stats:
        names30 = all_stats['D=30']['order']
        for n in names30:
            r10 = all_stats.get('D=10',{}).get('friedman',{}).get(n,'N/A')
            r30 = all_stats.get('D=30',{}).get('friedman',{}).get(n,'N/A')
            r50 = all_stats.get('D=50',{}).get('friedman',{}).get(n,'N/A')
            tag = '  ← PSGO' if n=='PSGO' else ''
            r10s = f"{r10:.4f}" if isinstance(r10,float) else str(r10)
            r30s = f"{r30:.4f}" if isinstance(r30,float) else str(r30)
            r50s = f"{r50:.4f}" if isinstance(r50,float) else str(r50)
            print(f"  {n:6s}  {r10s:8s}  {r30s:8s}  {r50s:8s}{tag}")

# Wilcoxon
if 'D=30' in all_stats:
    s = all_stats['D=30']['summary']
    print(f"\nWILCOXON (D=30): +{s['wins']} ={s['ties']} -{s['losses']} | Win rate: {s['winrate']}%")

# Engineering
if os.path.exists('/content/results/engineering.json'):
    eng = json.load(open('/content/results/engineering.json'))
    print("\nENGINEERING BEST VALUES:")
    for p,d in eng.items():
        psgo_b = d['algos'].get('PSGO',{}).get('best','N/A')
        print(f"  {p:20s}: PSGO best = {psgo_b}")

print("\n✅ ALL DONE!")
